# KG → LLM Evaluation — Local (CPU)

This notebook runs the evaluation pipeline **locally on your Mac (CPU only)**. No GPU, no Colab, no Neo4j cloud instance needed.

| Method | What it does | Runs locally? |
|--------|-------------|---------------|
| **Method 1** — SFT Data Quality | Structural audit, SFT pair generation, quality scoring | ✅ Yes |
| **Method 2** — Dataset Generation | Generate KG + raw-text QA datasets (no fine-tuning) | ✅ Yes |
| **Method 2** — Fine-Tuning | Train models on QA datasets | ❌ GPU required (use Colab) |

### Prerequisites
- Repo already cloned and `pip install -e ".[eval-model,neo4j]"` already run
- `.env` file at the repo root with your DeepSeek API key
- A `knowledge_graph.json` in `generated_KGs/output/` (or wherever your KG lives)

## ⚙️ Configuration

Adjust the KG path if yours is somewhere else.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ensure we're at the repo root (where .env lives)
_repo_root = Path(os.getcwd())
if not (_repo_root / ".env").exists():
    # Maybe we're in evaluation/model_eval/ — walk up
    for _ in range(3):
        _repo_root = _repo_root.parent
        if (_repo_root / ".env").exists():
            os.chdir(str(_repo_root))
            break

# Load credentials from .env
load_dotenv()

# ── Path to your knowledge graph ─────────────────────────────
KG_PATH = "generated_KGs/output/knowledge_graph.json"

# ── DeepSeek key check ───────────────────────────────────────
key = os.getenv("DEEPSEEK_API_KEY", "")
print(f"Repo root:     {os.getcwd()}")
print(f"KG path:       {KG_PATH}")
print(f"DeepSeek key:  {'***' + key[-4:] if len(key) > 4 else 'NOT SET (template generation only)'}")
print(f"Neo4j URI:     {os.getenv('NEO4J_URI', 'not set')}")

## Step 1: Method 1 — SFT Data Quality Assessment

Runs on CPU. Checks graph health, generates SFT training pairs, and evaluates their quality.

**⏱ ~30 seconds**

In [ ]:
import subprocess, json
from pathlib import Path

print("=" * 60)
print("METHOD 1: SFT Data Quality Assessment")
print("=" * 60)

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "1", "--kg", KG_PATH],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:2000])

# Show key results
method1_dir = Path("output_eval/method1")

audit_path = method1_dir / "structural_audit.json"
if audit_path.exists():
    with open(audit_path) as f:
        audit = json.load(f)
    print(f"\n📊 Structural Audit — Health Score: {audit.get('overall_health_score', '?')}/100")
    print(f"   Verdict: {audit.get('verdict', '?')}")

quality_path = method1_dir / "sft_quality_report.json"
if quality_path.exists():
    with open(quality_path) as f:
        quality = json.load(f)
    print(f"\n📊 SFT Quality — Overall Score: {quality.get('overall_score', '?'):.3f}")
    print(f"   Verdict: {quality.get('verdict', '?')}")

print(f"\n✅ Method 1 complete — full results in {method1_dir}/")

## Step 2: Auto-Generate Plots

Visualize the Method 1 results as charts.

In [ ]:
try:
    from generate_plots.plot_structural import plot_structural_audit
    from generate_plots.plot_sft_quality import plot_sft_quality

    audit_path = Path("output_eval/method1/structural_audit.json")
    quality_path = Path("output_eval/method1/sft_quality_report.json")
    output_dir = Path("output_eval/method1")

    if audit_path.exists():
        plot_structural_audit(audit_path, output_dir)
        print("✅ Structural audit plots generated")

    if quality_path.exists():
        plot_sft_quality(quality_path, output_dir)
        print("✅ SFT quality plots generated")

    print(f"\n📊 Plots saved to {output_dir}/")
except ImportError:
    print("⚠️  matplotlib not installed — skipping plots.")
    print("   Install with: pip install matplotlib")
except Exception as e:
    print(f"⚠️  Plot generation skipped: {e}")

## Step 3 (Optional): Generate QA Datasets for Fine-Tuning

Generates the KG-structured and raw-text QA datasets that Method 2 would use for fine-tuning. You can then take these to Colab to train models.

**⏱ ~10 seconds**

In [ ]:
import subprocess

print("=" * 60)
print("METHOD 2: Dataset Generation (no fine-tuning)")
print("=" * 60)
print("This generates QA pairs from your KG — useful to inspect or take to Colab.")

result = subprocess.run(
    [
        "python", "evaluation/run_eval.py",
        "--method", "2",
        "--kg", KG_PATH,
        "--skip-finetune",
        "--skip-benchmark",
    ],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:2000])

# Check generated files
output_dir = Path("output_eval/method2")
for name in ["kg_qa_train.jsonl", "kg_qa_test.jsonl", "raw_qa_train.jsonl", "raw_qa_test.jsonl"]:
    fpath = output_dir / name
    if fpath.exists():
        lines = sum(1 for _ in open(fpath))
        print(f"   ✅ {name}: {lines} pairs ({fpath.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"   ⚠️  {name}: not found")

print(f"\n📁 Datasets saved to {output_dir}/")
print("   Take these to Colab (run_eval_colab.ipynb) to fine-tune models.")

## Next: Fine-Tuning (GPU required)

To train models on these datasets, open [`run_eval_colab.ipynb`](run_eval_colab.ipynb) in Google Colab. Upload the generated QA datasets and follow the notebook.

Or use the CLI directly on a GPU machine:
```bash
python evaluation/run_eval.py --method 2 --kg generated_KGs/output/knowledge_graph.json \
    --fine-tune-target both --skip-dataset-gen
```